In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import logfire
logfire.configure()

Logfire project URL: ]8;id=12060326;https://logfire-us.pydantic.dev/eerga/starter-project\https://logfire-us.pydantic.dev/eerga/starter-project]8;;\


## Q1: Run the agent a few times with different questions and open your
project on Logfire to see the traces.

For the following query

> How do I run Ollama locally?

how many spans does a single agent run produce?

Each span is either the agent run itself, an LLM call, or a tool call.
The number can vary between runs because the model decides how many
times to search.

* 1
* 5
* 15
* 30


![screenshot](UI_screenshot.png)


## Q1 answer: 5

## Question 2. Load traces into DuckDB with dlt
How many tables did dlt create? Check with:

```sql
SELECT COUNT(*) FROM information_schema.tables 
WHERE table_schema = 'agent_traces';
```

* 1
* 3
* 24
* 100

ran `uv run python load_traces.py`

![screenshot](screen2.png)


## Q2 Answer: 24

## Question 3. Query traces with an agent

Using a coding agent (you can also write the code by hand) find the
input token usage for the agent run from Q1.

The token counts are stored in the span attributes as
`gen_ai.usage.input_tokens`. Sum them across all LLM calls within the
trace. The number depends on how many searches the agent made, so
report the range it falls into:

* 100 - 500
* 1500 - 5000
* 10000 - 20000
* 50000 - 100000

In [8]:
import duckdb

conn = duckdb.connect("logfire_pipeline.duckdb")

# Sum input tokens for all LLM calls in the Ollama trace
result = conn.execute("""
    SELECT 
        trace_id,
        SUM(attributes__gen_ai_usage_input_tokens) AS total_input_tokens
    FROM agent_traces.agent_traces
    WHERE span_name LIKE 'chat %'
      AND trace_id = '019f811c2140c62a1882d9d73ab2f550'
    GROUP BY trace_id
""").fetchone()

print(f"Total input tokens: {result[1]}")
conn.close()


Total input tokens: 4272


In [10]:
import duckdb

conn = duckdb.connect("logfire_pipeline.duckdb")

# Sum input tokens for all LLM calls in the Ollama trace
result = conn.execute("""
    SELECT 
        trace_id,
        SUM(attributes__gen_ai_usage_input_tokens) AS total_input_tokens
    FROM agent_traces.agent_traces
    WHERE span_name LIKE 'chat %'
      AND trace_id = '019f812493d2361e3c29b323ed65d57c'
    GROUP BY trace_id
""").fetchone()

print(f"Total input tokens: {result[1]}")
conn.close()


Total input tokens: 3859


## Q3 Answer: 1500 - 5000